# Data Preprocessing & Feature Engineering

## Objective

This notebook prepares the HR employee attrition dataset for machine learning by establishing a reusable, leakage-free preprocessing pipeline. We identify and remove constant and uninformative features, partition the data into stratified train and test sets, and construct a composite preprocessing pipeline using scikit-learn's `ColumnTransformer` to handle numerical scaling and categorical encoding.

# 1. Import Libraries

We import the required libraries for our preprocessing pipeline:
* `pandas` and `numpy` for data loading and array manipulation.
* `sklearn.model_selection.train_test_split` to partition the dataset.
* `sklearn.preprocessing.StandardScaler` and `OneHotEncoder` to transform the numerical and categorical features.
* `sklearn.pipeline.Pipeline` and `sklearn.compose.ColumnTransformer` to assemble our transformation stages.
* `joblib` to save the fitted pipeline for downstream model training and inference.

In [21]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.plot_config import *


# 2. Load Dataset

We load the raw HR employee dataset (`Employee_Data.csv`) which will be used to construct and validate our preprocessing pipeline.

In [22]:
df = pd.read_csv("../data/raw/Employee_Data.csv")
df.head()

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


# 3. Define Features and Target

We isolate the target column `Attrition` from the predictor features ($X$). We map the values of `Attrition` from categorical strings (`"Yes"` / `"No"`) to binary numeric values (`1` / `0`) to satisfy scikit-learn's input requirements.

> [!IMPORTANT]
> **Target Isolation**
> We isolate and encode the target variable independently of the feature columns to prevent target leakage during subsequent preprocessing steps.

In [23]:
X = df.drop(columns="Attrition")

y = df["Attrition"].map({
    "No":0,
    "Yes":1
})

# 4. Train-Test Split

We split the dataset into training (80%) and testing (20%) sets. This partition is performed before any data transformations to prevent data leakage from the test set into our scaling and encoding parameters.

In [24]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [25]:
print(X_train.shape)
print(X_test.shape)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

(1176, 34)
(294, 34)
Attrition
0    0.838435
1    0.161565
Name: proportion, dtype: float64
Attrition
0    0.840136
1    0.159864
Name: proportion, dtype: float64


> [!NOTE]
> **Why Stratified Sampling?**
>
> Employee attrition is an imbalanced classification problem, with employees who leave representing a smaller proportion of the dataset. Using **stratified sampling** ensures that both the training and testing datasets preserve the original class distribution. This leads to more reliable model evaluation and prevents bias caused by unequal representation of the target classes.

# 5. Feature Engineering

We identify and remove columns that do not contribute predictive value to the model:
1. **Constant columns**: Features with zero variance that hold the same value for every record.
2. **Database identifiers**: The `EmployeeNumber` column, which serves as a unique record key and contains no generalizable behavioral patterns.

Below is a summary of the features identified for removal:

| Feature | Reason |
| :--- | :--- |
| `EmployeeCount` | Constant feature (always contains $1$) |
| `StandardHours` | Constant feature (always contains $80$) |
| `Over18` | Constant feature (always contains `'Y'`) |
| `EmployeeNumber` | Unique identifier (database key) |

In [26]:
categorical_features = X_train.select_dtypes(
    include="str"
).columns.tolist()

numerical_features = X_train.select_dtypes(
    exclude="object"
).columns.tolist()

print(categorical_features)
print(numerical_features)

['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'Over18', 'OverTime']
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EmployeeCount', 'EmployeeNumber', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


### Preprocessing Cleaning Steps

We check the training set to programmatically identify columns containing only a single unique value.

In [27]:
constant_features = [
    col for col in X_train.columns
    if X_train[col].nunique() == 1
]

print("Constant Features:")
print(constant_features)

Constant Features:
['EmployeeCount', 'Over18', 'StandardHours']


### Drop Uninformative Columns

We drop the identified constant columns and the unique `EmployeeNumber` identifier from both the training and testing datasets.

In [28]:
drop_columns = constant_features + ["EmployeeNumber"]

X_train = X_train.drop(columns=drop_columns)
X_test = X_test.drop(columns=drop_columns)

print("Dropped Columns:")
print(drop_columns)

Dropped Columns:
['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']


In [29]:
print(f"Training Shape : {X_train.shape}")
print(f"Testing Shape  : {X_test.shape}")

Training Shape : (1176, 30)
Testing Shape  : (294, 30)


### Feature Engineering Summary

We have removed constant columns (`EmployeeCount`, `StandardHours`, `Over18`) and the unique key (`EmployeeNumber`) from the training and testing datasets. The dimensions have been reduced accordingly, and the datasets are now ready for feature type classification.

# 6. Feature Identification

We separate our remaining features into numerical and categorical lists to apply targeted preprocessing pipelines:
* **Numerical features** will be scaled to ensure a mean of 0 and a standard deviation of 1.
* **Categorical features** will be encoded into binary columns using one-hot representation.

In [30]:
# Update feature lists after dropping columns

categorical_features = (
    X_train.select_dtypes(include="str")
    .columns
    .tolist()
)

numerical_features = (
    X_train.select_dtypes(exclude="object")
    .columns
    .tolist()
)

# 7. Build Preprocessing Pipeline

We construct a scikit-learn preprocessing pipeline using `ColumnTransformer` to automate the distinct transformations required for our numerical and categorical features.

### Numerical Pipeline

We define a processing pipeline for the numerical columns, applying `StandardScaler` to normalize our numerical feature distributions.

In [31]:
numeric_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

numeric_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True


### Categorical Pipeline

We define a processing pipeline for our categorical features, applying `OneHotEncoder` to encode categorical text values. We set `handle_unknown='ignore'` to prevent errors during downstream inference when encountering new categories.

In [32]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

categorical_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('encoder', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used catego

In [33]:
numeric_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    (
        "encoder",
        OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        )
    )
])

### Column Transformer

We combine the numerical and categorical pipelines into a single `ColumnTransformer` object, mapping the numerical columns to our scaling pipeline and categorical columns to our encoding pipeline.

> [!TIP]
> **Transformer Modularity**
> By using `ColumnTransformer`, we consolidate all transformations into a single component. Updates to feature lists or preprocessing steps can be managed here without changes to downstream code.

In [34]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numerical_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# 8. Fit the Preprocessor

We fit the pipeline exclusively on the training dataset (`X_train`) to compute the scaling statistics (mean, variance) and extract categorical levels.

> [!WARNING]
> **Preventing Leakage**
> We do not fit the preprocessor on the test dataset or the full dataset. Fitting parameters must reside solely on the training partition to maintain strict target separation.

In [35]:
preprocessor.fit(X_train)

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``

# 9. Transform the Dataset

We transform both the training (`X_train`) and testing (`X_test`) sets using our fitted preprocessor, converting them into processed numpy arrays.

In [ ]:
X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 10. Verify the Processed Dataset

We verify the transformed datasets by performing checks on the shape of our arrays, examining the generated feature names, reviewing the top rows of the processed data as a pandas DataFrame, and confirming that no missing values are present in the final output.

In [37]:
print("Training Shape :", X_train_processed.shape)
print("Testing Shape  :", X_test_processed.shape)

Training Shape : (1176, 51)
Testing Shape  : (294, 51)


In [38]:
feature_names = preprocessor.get_feature_names_out()

print(f"Total Features: {len(feature_names)}")

feature_names[:20]

Total Features: 51


array(['num__Age', 'num__DailyRate', 'num__DistanceFromHome',
       'num__Education', 'num__EnvironmentSatisfaction',
       'num__HourlyRate', 'num__JobInvolvement', 'num__JobLevel',
       'num__JobSatisfaction', 'num__MonthlyIncome', 'num__MonthlyRate',
       'num__NumCompaniesWorked', 'num__PercentSalaryHike',
       'num__PerformanceRating', 'num__RelationshipSatisfaction',
       'num__StockOptionLevel', 'num__TotalWorkingYears',
       'num__TrainingTimesLastYear', 'num__WorkLifeBalance',
       'num__YearsAtCompany'], dtype=object)

In [39]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [42]:
X_train_processed_df.head(10)

,num__Age,num__DailyRate,num__DistanceFromHome,num__Education,num__EnvironmentSatisfaction,num__HourlyRate,num__JobInvolvement,num__JobLevel,num__JobSatisfaction,num__MonthlyIncome,...,cat__JobRole_Manufacturing Director,cat__JobRole_Research Director,cat__JobRole_Research Scientist,cat__JobRole_Sales Executive,cat__JobRole_Sales Representative,cat__MaritalStatus_Divorced,cat__MaritalStatus_Married,cat__MaritalStatus_Single,cat__OverTime_No,cat__OverTime_Yes
1194,1.090194,1.049455,-0.899915,1.064209,-0.658710,-0.908436,1.795282,1.762189,-0.647997,2.026752,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
128,-1.634828,-0.523449,-0.899915,-1.855332,0.260202,1.694111,0.373564,-0.986265,1.153526,-0.864408,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
810,0.981193,-0.992080,-0.777610,-1.855332,-1.577622,-0.662913,0.373564,1.762189,0.252765,2.347706,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
478,-1.307825,-0.453653,0.445433,-1.855332,-0.658710,-1.252169,0.373564,-0.986265,0.252765,-0.956202,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0
491,0.654191,0.491086,-0.043784,2.037390,1.179114,0.319180,0.373564,-0.070114,0.252765,-0.185956,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
323,-0.980822,0.879950,-0.899915,1.064209,-1.577622,0.908436,-2.469873,-0.986265,1.153526,-0.662120,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
258,1.526197,0.072310,-1.022219,0.091029,0.260202,1.497693,0.373564,-0.986265,1.153526,-0.821414,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
812,0.654191,0.692996,2.157694,0.091029,0.260202,0.859332,0.373564,0.846038,-1.548758,0.919216,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1132,0.327188,-1.709982,0.567738,-0.882152,1.179114,0.908436,0.373564,-0.070114,-1.548758,-0.409527,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
996,-1.089823,-1.493116,0.078520,0.091029,1.179114,1.595902,-1.048155,-0.070114,1.153526,-0.166609,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0


In [41]:
print(
    "Missing values (Train):",
    X_train_processed_df.isnull().sum().sum()
)

print(
    "Missing values (Test):",
    X_test_processed_df.isnull().sum().sum()
)

Missing values (Train): 0
Missing values (Test): 0


# 11. Save the Preprocessor

We save the fitted preprocessor pipeline to disk as a serialized `joblib` file. This allows us to reload and apply the exact same transformation metrics (scaling parameters and categorical categories) during modeling and model deployment.

In [43]:
from pathlib import Path
import joblib

save_dir = Path("../artifacts/preprocessors")
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    preprocessor,
    save_dir / "preprocessor.joblib"
)

print("✅ Preprocessor saved successfully.")

✅ Preprocessor saved successfully.


# 12. Verify Saved Pipeline

We load the saved pipeline back from disk to verify that serialization succeeded and that the preprocessor object is fully operational.

In [44]:
loaded_preprocessor = joblib.load(
    save_dir / "preprocessor.joblib"
)

print(type(loaded_preprocessor))

<class 'sklearn.compose._column_transformer.ColumnTransformer'>


# 13. Summary

## Key Accomplishments

* Structured the preprocessing workflow to prevent data leakage by splitting the dataset before applying scaling or encoding.
* Programmatically removed constant columns (`EmployeeCount`, `StandardHours`, `Over18`) and the unique record key (`EmployeeNumber`) from the training and testing sets.
* Constructed a composite column pipeline incorporating standard scaling for numerical features and one-hot encoding for categorical variables.
* Saved the fitted transformer object and verified its serialization integrity.

## Business Importance

Exporting our preprocessing steps as a unified, reusable pipeline guarantees that all predictions on incoming records (for example, monthly employee evaluations or new hires) are normalized using the identical parameters calculated during model training. This ensures model reliability and prevents mismatch issues when our predictor is deployed.

## Next Steps

In the next stage, we will utilize this preprocessing pipeline to train, tune, and compare classification models on the processed dataset.